In [10]:
# Cell 1: 导入模块和设置
import requests
import re
import time
from bs4 import BeautifulSoup
import csv

# 浏览器头
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.tvmaze.com/",
}
PROXIES = {"http": None, "https": None}

def clean_text(text):
    """去除多余空白"""
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print("✅ 环境加载完成")

✅ 环境加载完成


In [11]:
# Cell 2: 详细分析页面结构
def analyze_page_structure(url):
    """详细分析页面结构"""
    try:
        res = requests.get(url, headers=HEADERS, timeout=15, proxies=PROXIES)
        soup = BeautifulSoup(res.text, 'html.parser')
        
        print(f"🔍 分析页面: {url}")
        print("=" * 60)
        
        # 1. 查看所有包含信息的div的class
        print("1. 页面中所有的div class:")
        div_classes = set()
        for div in soup.find_all('div', class_=True):
            div_classes.add(div.get('class')[0] if div.get('class') else '')
        
        for cls in sorted(div_classes)[:20]:  # 只显示前20个
            print(f"   - {cls}")
        
        # 2. 查看所有包含信息的section
        print("\n2. 页面中所有的section:")
        for section in soup.find_all('section'):
            class_attr = section.get('class', [])
            print(f"   - class: {class_attr}")
            # 显示section内的前100个字符
            text = section.get_text(strip=True)[:100]
            if text:
                print(f"     内容: {text}...")
        
        # 3. 查看所有包含信息的article
        print("\n3. 页面中所有的article:")
        for article in soup.find_all('article'):
            class_attr = article.get('class', [])
            print(f"   - class: {class_attr}")
        
        # 4. 查看所有包含关键信息的元素
        print("\n4. 包含关键信息的元素:")
        keywords = ['Premiered', 'Ended', 'Status', 'Rating', 'Genre', 'Network', 'Summary']
        for keyword in keywords:
            elements = soup.find_all(string=re.compile(keyword, re.IGNORECASE))
            for element in elements[:2]:  # 只显示前2个匹配
                parent = element.parent
                print(f"   - {keyword}: {element.strip()}")
                if parent:
                    print(f"     父元素: {parent.name} class={parent.get('class')}")
        
        return soup
        
    except Exception as e:
        print(f"分析失败: {e}")
        return None

# 分析一个示例页面
test_url = "https://www.tvmaze.com/shows/60/ncis"
soup = analyze_page_structure(test_url)

🔍 分析页面: https://www.tvmaze.com/shows/60/ncis
1. 页面中所有的div class:
   - auto
   - callout
   - card
   - content
   - dropdown
   - flad
   - flad-300x250
   - grid-view
   - grid-x
   - header-wrap
   - hide-for-print
   - justwatch
   - left
   - medium-6
   - reveal
   - right
   - row
   - show-for-small-only
   - small
   - small-12

2. 页面中所有的section:
   - class: ['grid-x', 'grid-padding-x', 'margin-bottom']
   - class: []
     内容: FollowFollowingNCIS(Naval Criminal Investigative Service) is more than just an action drama. With li...
   - class: ['row']
     内容: FollowFollowingNCIS(Naval Criminal Investigative Service) is more than just an action drama. With li...
   - class: ['small-12', 'medium-8', 'columns', 'row']
     内容: FollowFollowingNCIS(Naval Criminal Investigative Service) is more than just an action drama. With li...
   - class: []
     内容: Watch now...
   - class: ['small-12', 'medium-4', 'columns']
     内容: Show InfoNetwork:CBS(2003                            -
       

In [16]:
# Cell 3: 修复的解析器
def parse_show_page_fixed(url):
    """修复的解析器 - 准确提取所有字段"""
    try:
        res = requests.get(url, headers=HEADERS, timeout=15, proxies=PROXIES)
        if res.status_code != 200:
            return None
            
        soup = BeautifulSoup(res.text, 'html.parser')
        
        result = {
            "Title": "",
            "First air date": "",
            "End date": "",
            "Rating": "",
            "Genres": "",
            "Status": "",
            "Network": "",
            "Summary": "",
            "URL": url
        }
        
        # 1. 提取标题
        title_elem = soup.find('h1')
        if title_elem:
            result["Title"] = clean_text(title_elem.get_text())
        
        # 2. 查找主要信息区域 - callout section
        info_section = soup.find('section', class_='callout')
        if info_section:
            # 获取所有文本行
            lines = [clean_text(line) for line in info_section.get_text().split('\n') if clean_text(line)]
            
            print(f"🔍 调试信息 - {result['Title']}:")  # 调试信息
            for i, line in enumerate(lines):
                print(f"   行 {i}: {line}")
            
            for line in lines:
                # 网络/频道信息 - 修复提取逻辑
                if not result["Network"] and ('Network:' in line or 'Web channel:' in line):
                    if 'Network:' in line:
                        network_match = re.search(r'Network:\s*(.+)', line)
                    else:
                        network_match = re.search(r'Web channel:\s*(.+)', line)
                    
                    if network_match:
                        network_text = clean_text(network_match.group(1))
                        # 提取网络名称（去掉年份信息）
                        network_name = re.split(r'\(\d', network_text)[0].strip()
                        result["Network"] = network_name
                        
                        # 从网络信息中提取首播年份 - 修复：只提取开始年份
                        year_match = re.search(r'\((\d{4})\s*[-–]', network_text)
                        if year_match:
                            result["First air date"] = year_match.group(1)
                            # 这里不能提取结束年份，因为正则表达式只有一个捕获组
                
                # 状态信息 - 修复提取逻辑
                if not result["Status"] and 'Status:' in line:
                    status_match = re.search(r'Status:\s*(.+)', line)
                    if status_match:
                        result["Status"] = clean_text(status_match.group(1))
                
                # 类型信息 - 修复提取逻辑
                if not result["Genres"] and 'Genres:' in line:
                    genre_match = re.search(r'Genres:\s*(.+)', line)
                    if genre_match:
                        result["Genres"] = clean_text(genre_match.group(1))
                
                # 评分信息
                if not result["Rating"]:
                    # 匹配 "8.5(722 votes)" 这种格式
                    rating_match = re.search(r'(\d+\.?\d*)\s*\(\s*(\d+)\s*votes?\s*\)', line)
                    if rating_match:
                        rating_value = rating_match.group(1)
                        votes_count = rating_match.group(2)
                        result["Rating"] = f"{rating_value} ({votes_count} votes)"
        
        # 3. 如果没有找到年份，尝试其他方式 - 修复：使用正确的正则表达式
        if not result["First air date"] or not result["End date"]:
            # 在整个信息区域查找年份
            if info_section:
                info_text = info_section.get_text()
                # 修复：使用有两个捕获组的正则表达式
                year_match = re.search(r'\((\d{4})\s*[-–]\s*(\d{4}|now|present)\)', info_text)
                if year_match:
                    result["First air date"] = year_match.group(1)
                    if year_match.group(2) not in ['now', 'present']:
                        result["End date"] = year_match.group(2)
                    else: 
                        result["End date"] = "Not ended"
                   
        # 4. 如果没有找到Genres，尝试其他方式
        if not result["Genres"] and info_section:
            info_text = info_section.get_text()
            genre_match = re.search(r'Genres:\s*([^\n]+)', info_text)
            if genre_match:
                result["Genres"] = clean_text(genre_match.group(1))
        
        # 5. 提取摘要
        if not result["Summary"]:
            # 查找描述性内容
            all_sections = soup.find_all('section')
            for section in all_sections:
                if 'callout' not in section.get('class', []):
                    text = clean_text(section.get_text())
                    # 检查是否是剧情描述
                    if (len(text) > 80 and len(text) < 1500 and
                        not any(keyword in text for keyword in ['Network:', 'Status:', 'Genres:', 'Rating:', 'Show Info', 'FollowFollowing']) and
                        not re.search(r'Season \d+', text) and
                        not re.search(r'Episode \d+', text) and
                        not re.search(r'\d+\.?\d*\s*\(\d+\s*votes?\)', text)):
                        result["Summary"] = text
                        break
        
        return result
        
    except Exception as e:
        print(f"❌ 解析失败 {url}: {e}")
        return None

# 测试修复的解析器
print("🧪 测试修复的解析器...")
test_url = "https://www.tvmaze.com/shows/2993/stranger-things"
data = parse_show_page_fixed(test_url)
if data:
    print(f"\n✅ 修复后的结果:")
    print(f"标题: {data['Title']}")
    all_fields = ["First air date", "End date", "Status", "Genres", "Network", "Rating", "Summary"]
    for key in all_fields:
        if data[key]:
            display_value = data[key]
            if key == "Summary" and len(display_value) > 100:
                display_value = display_value[:100] + "..."
            print(f"{key}: {display_value}")
        else:
            print(f"{key}: [空]")
else:
    print("❌ 解析失败")

🧪 测试修复的解析器...
🔍 调试信息 - Stranger Things:
   行 0: Show Info
   行 1: Web channel: Netflix
   行 2: (2016 -
   行 3: now)
   行 4: Average Runtime: 70 minutes
   行 5: Status: Running; returning November 2025
   行 6: Show Type:
   行 7: Scripted
   行 8: Genres:
   行 9: DramaHorrorScience-Fiction
   行 10: Episodes ordered: 8 episodes
   行 11: Created by:
   行 12: Ross DufferMatt Duffer
   行 13: Official site: www.netflix.com
   行 14: 8.5 (722 votes)

✅ 修复后的结果:
标题: Stranger Things
First air date: 2016
End date: Not ended
Status: Running; returning November 2025
Genres: DramaHorrorScience-Fiction
Network: Netflix
Rating: 8.5 (722 votes)
Summary: Follow Following When a young boy vanishes, a small town uncovers a mystery involving secret experim...


In [17]:
# Cell 4: 获取剧集链接
def get_show_links_optimized(page_url):
    """获取剧集链接"""
    try:
        res = requests.get(page_url, headers=HEADERS, timeout=15, proxies=PROXIES)
        if res.status_code != 200:
            return []
            
        soup = BeautifulSoup(res.text, 'html.parser')
        show_links = set()  # 使用set自动去重
        
        # 主要方法：查找包含剧集信息的卡片或列表项
        selectors_to_try = [
            'a[href*="/shows/"]',  # 所有包含/shows/的链接
            '.card a[href*="/shows/"]',
            '.show-card a[href*="/shows/"]',
            'article a[href*="/shows/"]',
            '.list-item a[href*="/shows/"]',
            '.tvshow a[href*="/shows/"]'
        ]
        
        for selector in selectors_to_try:
            links = soup.select(selector)
            for link in links:
                href = link.get('href')
                if href and '/shows/' in href:
                    if not href.startswith('http'):
                        href = "https://www.tvmaze.com" + href
                    # 确保是剧集详情页，不是其他页面
                    if re.match(r'https://www.tvmaze.com/shows/\d+/[^/]+', href):
                        show_links.add(href)
        
        return list(show_links)
        
    except Exception as e:
        print(f"获取链接失败: {e}")
        return []

print("🔍 开始获取剧集链接...")
all_links = []
seen_ids = set()  # 用ID去重

for page in range(3):  # 抓取3页
    list_url = f"https://www.tvmaze.com/shows?page={page}"
    print(f"📄 获取第 {page} 页...")
    
    page_links = get_show_links_optimized(list_url)
    print(f"   找到 {len(page_links)} 个链接")
    
    # 进一步去重：按剧集ID去重
    for link in page_links:
        # 提取剧集ID，如 /shows/28276/the-witcher 中的 28276
        match = re.search(r'/shows/(\d+)/', link)
        if match:
            show_id = match.group(1)
            if show_id not in seen_ids:
                seen_ids.add(show_id)
                all_links.append(link)
    
    time.sleep(1)  # 延迟

print(f"✅ 去重后总共找到 {len(all_links)} 个唯一剧集链接")

# 显示前几个链接作为检查
print("\n前10个链接示例:")
for i, link in enumerate(all_links[:10]):
    print(f"  {i+1}. {link}")

🔍 开始获取剧集链接...
📄 获取第 0 页...
   找到 25 个链接
📄 获取第 1 页...
   找到 25 个链接
📄 获取第 2 页...
   找到 25 个链接
✅ 去重后总共找到 50 个唯一剧集链接

前10个链接示例:
  1. https://www.tvmaze.com/shows/112/south-park
  2. https://www.tvmaze.com/shows/28152/9-1-1
  3. https://www.tvmaze.com/shows/32158/fbi
  4. https://www.tvmaze.com/shows/49964/invasion
  5. https://www.tvmaze.com/shows/49041/fallout
  6. https://www.tvmaze.com/shows/2246/chicago-med
  7. https://www.tvmaze.com/shows/28276/the-witcher
  8. https://www.tvmaze.com/shows/67/greys-anatomy
  9. https://www.tvmaze.com/shows/48830/only-murders-in-the-building
  10. https://www.tvmaze.com/shows/62998/gen-v


In [27]:
# Cell 5: 使用parse_show_page_fixed函数爬取数据
print("🔄 使用修复的解析器重新爬取数据...")
all_data = []
total = len(all_links)

for i, url in enumerate(all_links, 1):
    print(f"🔍 ({i}/{total}) 解析: {url}")
    
    # 使用Cell 3中定义的parse_show_page_fixed函数
    data = parse_show_page_fixed(url)
    if data and data["Title"]:
        all_data.append(data)
        
        # 显示提取到的信息 - 使用和测试一样的格式
        print(f"✅ {data['Title']}")
        all_fields = ["First air date", "End date", "Status", "Genres", "Network", "Rating", "Summary"]
        for key in all_fields:
            if data[key]:
                display_value = data[key]
                if key == "Summary" and len(display_value) > 100:
                    display_value = display_value[:100] + "..."
                print(f"   {key}: {display_value}")
            else:
                print(f"   {key}: [空]")
    else:
        print(f"⚠️ 解析失败")
    
    time.sleep(1)
    
    # 每10个显示一次进度
    if i % 10 == 0:
        completed_rating = len([d for d in all_data if d["Rating"]])
        completed_enddate = len([d for d in all_data if d["End date"]])
        completed_summary = len([d for d in all_data if d["Summary"]])
        print(f"📊 进度: {i}/{total} | 有评分的: {completed_rating} | 有结束日期的: {completed_enddate} | 有摘要的: {completed_summary}")

print(f"📦 完成！共收集 {len(all_data)} 条记录")

🔄 使用修复的解析器重新爬取数据...
🔍 (1/50) 解析: https://www.tvmaze.com/shows/112/south-park
🔍 调试信息 - South Park:
   行 0: Show Info
   行 1: Network: Comedy Central
   行 2: (1997 -
   行 3: now)
   行 4: Schedule: Wednesdays at 22:00
   行 5: (30 min)
   行 6: Status: Running
   行 7: Show Type:
   行 8: Animation
   行 9: Genres:
   行 10: Comedy
   行 11: Episodes ordered: 5 episodes
   行 12: Created by:
   行 13: Trey ParkerMatt Stone
   行 14: Official site: www.cc.com
   行 15: 8.3 (273 votes)
✅ South Park
   First air date: 1997
   End date: Not ended
   Status: Running
   Genres: Comedy
   Network: Comedy Central
   Rating: 8.3 (273 votes)
   Summary: Follow Following South Park is an adult comedy animation show centred around 4 children in the small...
🔍 (2/50) 解析: https://www.tvmaze.com/shows/28152/9-1-1
🔍 调试信息 - 9-1-1:
   行 0: Show Info
   行 1: Network: ABC
   行 2: (2018 -
   行 3: now)
   行 4: Schedule: Thursdays at 20:00
   行 5: (60 min)
   行 6: Status: Running
   行 7: Show Type:
   行 8: Scripted
   行 9

In [29]:
# Cell 6: 保存数据
def save_to_csv(data, filename):
    """保存数据到CSV文件"""
    if not data:
        print("❌ 没有数据可保存")
        return False
    
    try:
        fieldnames = data[0].keys()
        
        with open(filename, 'w', newline='', encoding='utf-8-sig') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)
        
        print(f"💾 已保存 {len(data)} 条记录到 {filename}")
        return True
        
    except Exception as e:
        print(f"❌ 保存文件失败: {e}")
        return False

# 保存数据
OUTPUT_CSV = "LaiYiming_2254030.csv"
if save_to_csv(all_data, OUTPUT_CSV):
    # 统计结果
    print(f"\n📈 字段统计 (共 {len(all_data)} 条记录):")
    print("-" * 50)
    
    field_stats = {}
    for item in all_data:
        for key, value in item.items():
            if key not in field_stats:
                field_stats[key] = 0
            if value and str(value).strip():
                field_stats[key] += 1
    
    for key, count in field_stats.items():
        percentage = (count / len(all_data)) * 100
        print(f"  {key:<15}: {count:>2}/{len(all_data)} ({percentage:5.1f}%)")
    
    # 显示有详细信息的记录
    detailed_records = []
    for item in all_data:
        detailed_fields = sum(1 for field in ["First air date", "End date", "Rating", "Genres", "Status", "Network", "Summary"] 
                            if item[field] and str(item[field]).strip())
        if detailed_fields >= 2:  # 至少有2个详细信息字段
            detailed_records.append(item)
    
    print(f"\n🎯 有详细信息的记录: {len(detailed_records)}/{len(all_data)}")
    
    # 预览前3条记录
    print(f"\n📊 数据预览 (前3条):")
    print("=" * 60)
    for i, item in enumerate(all_data[:3]):
        print(f"\n--- 记录 {i+1} ---")
        for key, value in item.items():
            if value and str(value).strip():
                display_value = str(value)
                if len(display_value) > 100:
                    display_value = display_value[:100] + "..."
                print(f"  {key}: {display_value}")
    
    print(f"\n💾 文件已保存为: {OUTPUT_CSV}")
    
else:
    print("❌ 保存失败")

💾 已保存 50 条记录到 LaiYiming_2254030.csv

📈 字段统计 (共 50 条记录):
--------------------------------------------------
  Title          : 50/50 (100.0%)
  First air date : 50/50 (100.0%)
  End date       : 50/50 (100.0%)
  Rating         : 50/50 (100.0%)
  Genres         : 50/50 (100.0%)
  Status         : 50/50 (100.0%)
  Network        : 50/50 (100.0%)
  Summary        : 50/50 (100.0%)
  URL            : 50/50 (100.0%)

🎯 有详细信息的记录: 50/50

📊 数据预览 (前3条):

--- 记录 1 ---
  Title: South Park
  First air date: 1997
  End date: Not ended
  Rating: 8.3 (273 votes)
  Genres: Comedy
  Status: Running
  Network: Comedy Central
  Summary: Follow Following South Park is an adult comedy animation show centred around 4 children in the small...
  URL: https://www.tvmaze.com/shows/112/south-park

--- 记录 2 ---
  Title: 9-1-1
  First air date: 2018
  End date: Not ended
  Rating: 7.5 (239 votes)
  Genres: DramaActionCrime
  Status: Running
  Network: ABC
  Summary: Follow Following 9-1-1 is a fast-paced exploration

In [30]:
import os
print("当前工作目录:", os.getcwd())

当前工作目录: C:\Users\Administrator
